In [1]:
import sys
!{sys.executable} -m pip install scikit-learn

   ---------------------------------------- 0.0/8.2 MB ? eta -:--:--
   ---------------------------------------- 0.0/8.2 MB ? eta -:--:--
   - -------------------------------------- 0.3/8.2 MB ? eta -:--:--
   -- ------------------------------------- 0.5/8.2 MB 1.1 MB/s eta 0:00:07
   -- ------------------------------------- 0.5/8.2 MB 1.1 MB/s eta 0:00:07
   --- ------------------------------------ 0.8/8.2 MB 933.1 kB/s eta 0:00:08
   --- ------------------------------------ 0.8/8.2 MB 933.1 kB/s eta 0:00:08
   ----- ---------------------------------- 1.0/8.2 MB 761.4 kB/s eta 0:00:10
   ----- ---------------------------------- 1.0/8.2 MB 761.4 kB/s eta 0:00:10
   ------ --------------------------------- 1.3/8.2 MB 734.0 kB/s eta 0:00:10
   ------- -------------------------------- 1.6/8.2 MB 741.1 kB/s eta 0:00:09
   -------- ------------------------------- 1.8/8.2 MB 829.8 kB/s eta 0:00:08
   ---------- ----------------------------- 2.1/8.2 MB 875.8 kB/s eta 0:00:07
   ----------- --


[notice] A new release of pip is available: 25.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import os
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# 1. Muat data tabel CSV 60 kasus KDRT yang sudah kamu buat tadi
csv_path = "../data/processed/cases.csv"
if not os.path.exists(csv_path):
    print(" Waduh, file CSV tidak ditemukan! Pastikan Tahap 2 sudah sukses.")
else:
    df = pd.read_csv(csv_path)
    print(f"Sukses memuat {len(df)} kasus lama dari database CSV!\n")

    # 2. INTI CBR: Menghitung Bobot Kata menggunakan TF-IDF
    # Kita ubah kolom 'fakta_hukum' menjadi matriks angka agar bisa dihitung kemiripannya
    vectorizer = TfidfVectorizer()
    tfidf_matrix = vectorizer.fit_transform(df['fakta_hukum'].fillna(''))

    # 3. FUNGSI RETRIEVAL: Mencari Kasus Paling Mirip
    def cari_kasus_mirip(kasus_baru, top_n=3):
        # Ubah teks kasus baru menjadi bentuk angka (vektor)
        query_vector = vectorizer.transform([kasus_baru.lower()])
        
        # Hitung skor kemiripan antara kasus baru dengan 60 kasus lama (Cosine Similarity)
        skor_kemiripan = cosine_similarity(query_vector, tfidf_matrix).flatten()
        
        # Tambahkan kolom skor ke dalam dataframe sementara
        df_hasil = df.copy()
        df_hasil['skor_kemiripan'] = skor_kemiripan
        
        # Urutkan dari yang skornya paling tinggi (paling mirip)
        df_hasil = df_hasil.sort_values(by='skor_kemiripan', ascending=False)
        
        # Ambil sejumlah top_n teratas
        return df_hasil.head(top_n)

    # --- SIMULASI PENGUJIAN SISTEM CBR KAMU ---
    # Bayangkan ini adalah input kasus KDRT baru yang ingin dicari solusinya
    kasus_inputan_baru = "Terdakwa melakukan kekerasan fisik memukul istri karena masalah keuangan keluarga"
    
    print(f"--- INPUT KASUS BARU ---")
    print(f"Deskripsi: '{kasus_inputan_baru}'\n")
    print(f"--- MEMULAI PROSES RETRIEVAL (PENCARIAN KASUS MIRIP) ---")
    
    Hasil_pencarian = cari_kasus_mirip(kasus_inputan_baru, top_n=3)
    
    # Tampilkan hasil ke layar
    for idx, row in Hasil_pencarian.iterrows():
        print(f"\n Rank: {row['id_kasus']} (Skor Kemiripan: {row['skor_kemiripan']:.4f})")
        print(f" Nomor Perkara  : {row['nomor_perkara']}")
        print(f" Pasal Dilanggar: {row['pasal_dilanggar']}")
        print(f" Potongan Fakta : {row['fakta_hukum'][:150]}...")
        print("-" * 50)

Sukses memuat 60 kasus lama dari database CSV!

--- INPUT KASUS BARU ---
Deskripsi: 'Terdakwa melakukan kekerasan fisik memukul istri karena masalah keuangan keluarga'

--- MEMULAI PROSES RETRIEVAL (PENCARIAN KASUS MIRIP) ---

 Rank: CASE_004 (Skor Kemiripan: 0.2600)
 Nomor Perkara  : 23
 Pasal Dilanggar: PASAL 44 UU KDRT (DEFAULT)
 Potongan Fakta : Terdakwa terbukti melakukan kekerasan fisik/psikis dalam lingkup rumah tangga sebagaimana fakta persidangan....
--------------------------------------------------

 Rank: CASE_003 (Skor Kemiripan: 0.2600)
 Nomor Perkara  : 23
 Pasal Dilanggar: PASAL 44 UU KDRT (DEFAULT)
 Potongan Fakta : Terdakwa terbukti melakukan kekerasan fisik/psikis dalam lingkup rumah tangga sebagaimana fakta persidangan....
--------------------------------------------------

 Rank: CASE_007 (Skor Kemiripan: 0.2600)
 Nomor Perkara  : 1-K/PM.I-07/AD/I/2025DEMI
 Pasal Dilanggar: PASAL 44 UU KDRT (DEFAULT)
 Potongan Fakta : Terdakwa terbukti melakukan kekerasan fisik/psi